In [1]:
!pip install -q --prefer-binary \
    "langchain>=0.2.16,<0.4" \
    "langchain-community>=0.2.16,<0.4" \
    "langchain-core>=0.2.40,<0.4" \
    "langchain-text-splitters" \
    "langchain-huggingface" \
    chromadb pypdf sentence-transformers bitsandbytes accelerate pydantic pandas gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
langgraph-sdk 0.4.12 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but 

In [2]:
import os
import re
import json
import shutil
import urllib.request
import warnings
import logging
import torch
import nltk
import pandas as pd
import gradio as gr
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_text_splitters import NLTKTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.tools import tool
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
# Configuração do repositório GitHub para download automático no Colab
GITHUB_USER = "victro666"
GITHUB_REPO = "AgenteIA_Extrator_CFBs_APF"
GITHUB_BRANCH = "main"
GITHUB_RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}"

CACHE_DIR = "./huggingface_cache"
DATA_DIR = "./dados"
OUTPUT_DIR = "./resultados"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAMINHO_REGRAS_CPM = os.path.join(DATA_DIR, "regras_CPM_v2.txt")
CAMINHO_DOC_EXEMPLO = os.path.join(DATA_DIR, "0000_cctns_pt.txt")

arquivos_necessarios = [
    ("regras_CPM_v2.txt", CAMINHO_REGRAS_CPM),
    ("0000_cctns_pt.txt", CAMINHO_DOC_EXEMPLO)
]

for nome_arq, caminho_local in arquivos_necessarios:
    if not os.path.exists(caminho_local):
        url_arquivo = f"{GITHUB_RAW_URL}/dados/{nome_arq}"
        print(f"📥 Baixando '{nome_arq}' do repositório GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
            print(f"✅ '{nome_arq}' salvo em {caminho_local}")
        except Exception as e:
            print(f"⚠️ Não foi possível baixar '{nome_arq}' automaticamente ({e}). Faça upload manual na pasta '{DATA_DIR}'.")
    else:
        print(f"✅ Arquivo '{nome_arq}' já disponível localmente em {caminho_local}.")

📥 Baixando 'regras_CPM_v2.txt' do repositório GitHub...
✅ 'regras_CPM_v2.txt' salvo em ./dados/regras_CPM_v2.txt
📥 Baixando '0000_cctns_pt.txt' do repositório GitHub...
✅ '0000_cctns_pt.txt' salvo em ./dados/0000_cctns_pt.txt


In [ ]:
# CARREGAMENTO DO MODELO QWEN 2.5 (7B) EM 8-BITS
warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR

print("Carregando Qwen 2.5 (7B) Instruct em 8-bits...")
model_id = "Qwen/Qwen2.5-7B-Instruct"
quant_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)

# Pipeline determinístico
llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    do_sample=False,
    return_full_text=False
)

llm_base = HuggingFacePipeline(pipeline=llm_pipeline)
chat_llm = ChatHuggingFace(llm=llm_base)
print("✅ Modelo carregado e pronto!")

Carregando Qwen 2.5 (7B) Instruct em 8-bits...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (7 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Modelo carregado e pronto!


In [ ]:
# BASE VETORIAL DO MANUAL CPM (CHROMA DB)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Construindo a base vetorial do CPM...")
docs_regras = TextLoader(CAMINHO_REGRAS_CPM, encoding="utf-8").load()

text_splitter_regras = NLTKTextSplitter(chunk_size=1000, chunk_overlap=150)
splits_regras = text_splitter_regras.split_documents(docs_regras)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore_regras = Chroma.from_documents(documents=splits_regras, embedding=embeddings)
retriever_cpm = vectorstore_regras.as_retriever(search_kwargs={"k": 3})

# Transformação do ChromaDB em uma Tool que o Agente pode consultar sob demanda
@tool
def consultar_manual_cpm(duvida_classificacao: str) -> str:
    """Consulta o Manual de Práticas de Contagem (CPM) do IFPUG para tirar dúvidas sobre regras de EE, SE, CE, ALI ou AIE."""
    docs = retriever_cpm.invoke(duvida_classificacao)
    return "\n\n".join([doc.page_content for doc in docs])

print("✅ Ferramenta 'consultar_manual_cpm' registrada!")

Construindo a base vetorial do CPM...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Ferramenta 'consultar_manual_cpm' registrada!


In [6]:
# ESTRUTURAÇÃO DE SAÍDA COM PYDANTIC E CHAIN-OF-THOUGHT

# 1. Esquema para a decisão autônoma do Agente (Triagem + Tool Calling)
class DecisaoAgente(BaseModel):
    raciocinio_triagem: str = Field(
        description="Passo 1 (CoT): Identifique se o trecho descreve funcionalidades de negócio/dados de domínio ou apenas requisitos não funcionais (performance, hardware, usabilidade, login/senha, cookies, plugins, formatação física ASCII)."
    )
    possui_requisito_funcional: bool = Field(
        description="Passo 2: True se houver transações de usuário, sincronização/importação/exportação de dados ou entidades de dados. False se for puramente Não Funcional, sumário ou restrição física."
    )
    precisa_consultar_manual: bool = Field(
        description="Passo 3: True APENAS se houver dúvida entre SE e CE, compartilhamento de dados entre sistemas ou fronteira ambígua. False para casos claros."
    )
    query_busca_manual: Optional[str] = Field(
        default=None,
        description="Passo 4: Dúvida técnica para buscar no vetor do CPM caso precisa_consultar_manual seja True."
    )

# 2. Esquema para cada Componente Funcional Básico (CFB)
class ComponenteFuncional(BaseModel):
    trecho_referencia: str = Field(
        description="Passo 1 (Evidência): Citação exata do fragmento do texto que comprova este componente."
    )
    raciocinio_cpm: str = Field(
        description="Passo 2 (CoT): Justifique pelo CPM. Para transações: é completa, autocontida e deixa o sistema em estado consistente? Qual a intenção primária (EE, SE ou CE)? Para dados: é mantido internamente (ALI) ou apenas lido de outro sistema (AIE)?"
    )
    nome_processo: str = Field(
        description="Passo 3 (Nome Único e Atômico): PROIBIDO usar vírgulas ou agrupar itens! Para transações, use 1 verbo no infinitivo + 1 substantivo singular (ex: 'Registrar Aula', NUNCA 'Registrar Aula, Falta e Nota'). Para ALI/AIE, use apenas o substantivo da entidade no plural (ex: 'Aulas', 'Notas', 'Turmas')."
    )
    classificacao: Literal["EE", "SE", "CE", "ALI", "AIE"] = Field(
        description="Passo 4: Classificação estrita segundo o IFPUG."
    )
    confianca: Literal["Alta", "Media", "Baixa"] = Field(
        description="Passo 5: Grau de certeza na extração."
    )

class ListaExtracao(BaseModel):
    analise_preliminar_do_trecho: str = Field(
        description="Passo 0: Liste primeiro quais ENTIDADES DE DADOS (ALIs/AIEs) são manipuladas ou lidas neste trecho e depois quais TRANSAÇÕES COMPLETAS (EE, SE, CE) existem, descartando sub-passos de tela e requisitos não funcionais."
    )
    componentes: List[ComponenteFuncional] = Field(
        default_factory=list,
        description="Lista de CFBs atômicos (1 objeto por CFB). Sempre inclua tanto as Funções de Transação quanto as Funções de Dados (ALI/AIE) identificadas no trecho."
    )

parser_decisao = PydanticOutputParser(pydantic_object=DecisaoAgente)
parser_extracao = PydanticOutputParser(pydantic_object=ListaExtracao)

# Tabela determinística de pesos (NESMA Estimativa)
PESOS_NESMA_ESTIMATIVA = {
    "EE": 4,
    "SE": 5,
    "CE": 4,
    "ALI": 7,
    "AIE": 5
}

In [7]:
# CARREGAMENTO DE DOCUMENTOS E SUMARIZAÇÃO DE CONTEXTO GLOBAL
def carregar_e_fatiar_documento(caminho_arquivo: str):
    """Lê um arquivo .pdf ou .txt, limpa quebras de linha e divide em chunks de sentenças."""
    extensao = os.path.splitext(caminho_arquivo)[1].lower()
    if extensao == '.pdf':
        docs_brutos = PyPDFLoader(caminho_arquivo).load()
    elif extensao == '.txt':
        docs_brutos = TextLoader(caminho_arquivo, encoding="utf-8").load()
    else:
        raise ValueError(f"Formato não suportado: {extensao}. Use .pdf ou .txt.")

    texto_limpo_completo = ""
    for doc in docs_brutos:
        texto_processado = re.sub(r'\s+', ' ', doc.page_content)
        texto_limpo_completo += texto_processado + "\n"

    text_splitter_req = NLTKTextSplitter(chunk_size=1200, chunk_overlap=150)
    splits = text_splitter_req.create_documents([texto_limpo_completo])
    return texto_limpo_completo, splits

In [8]:
def gerar_contexto_global(texto_completo: str) -> str:
    """Extrai o resumo executivo com Objetivo, Atores e Entidades de Dados do sistema."""
    amostra_global = texto_completo[:6000]
    prompt_sumarizacao = f"""<|im_start|>system
Você é um Arquiteto de Software e Analista CFPS.
Analise a visão geral do documento de requisitos abaixo e resuma de forma ultra-compacta:
1. Objetivo principal do sistema.
2. Principais Atores/Usuários.
3. Principais Entidades de Dados (potenciais ALIs/AIEs como Clientes, Produtos, Pedidos).<|im_end|>
<|im_start|>user
[Documento]:
{amostra_global}<|im_end|>
<|im_start|>assistant
"""
    return llm_base.invoke(prompt_sumarizacao).strip()

# Executa no documento de exemplo baixado do GitHub
print(f"Carregando e limpando o documento de exemplo: {CAMINHO_DOC_EXEMPLO}...")
texto_limpo_completo, splits_requisitos = carregar_e_fatiar_documento(CAMINHO_DOC_EXEMPLO)

print("Gerando Mapa Global de Contexto (Atores e Entidades do Sistema)...")
contexto_global_sistema = gerar_contexto_global(texto_limpo_completo)
print("\n📌 CONTEXTO GLOBAL EXTRAÍDO:")
print(contexto_global_sistema)
print("-" * 60)

Carregando e limpando o documento de exemplo: ./dados/0000_cctns_pt.txt...
Gerando Mapa Global de Contexto (Atores e Entidades do Sistema)...

📌 CONTEXTO GLOBAL EXTRAÍDO:
**Resumo Ultra-Compacto:**

1. **Objetivo Principal:** Agregar valor ao pessoal policial na linha de frente, melhorando investigações e detecção de criminosos.
2. **Principais Atores/Usuários:** Policiais, cidadãos, tribunais.
3. **Entidades de Dados:** Queixas, investigações, ações penais, pesquisas, interações com cidadãos.
------------------------------------------------------------


In [9]:
# PROMPTS FEW-SHOT, CHAIN-OF-THOUGHT E MOTOR NEURO-SIMBÓLICO

EXEMPLOS_FEW_SHOT_JSON = """
### EXEMPLO 1 (Decomposição de 'Manter/Gerenciar' + Extração Obrigatória de ALI):
[Trecho do Requisito]: "O sistema deve permitir que o gerente mantenha o cadastro de fornecedores (informando CNPJ e endereço) e emita o relatório consolidado exibindo o cálculo do total gasto por fornecedor."
[Saída Esperada]:
{
  "analise_preliminar_do_trecho": "Entidade de dados identificada: Fornecedores (mantida pelo sistema -> ALI). Transações: O termo macro 'mantenha o cadastro' decompõe-se nas transações elementares de Incluir, Alterar, Excluir e Consultar Fornecedor. Há também um relatório com cálculo matemático (SE).",
  "componentes": [
    {
      "trecho_referencia": "mantenha o cadastro de fornecedores (informando CNPJ e endereço)",
      "raciocinio_cpm": "Grupo lógico de dados de negócio reconhecido pelo usuário e mantido dentro da fronteira da aplicação através de processos elementares.",
      "nome_processo": "Fornecedores",
      "classificacao": "ALI",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "mantenha o cadastro de fornecedores",
      "raciocinio_cpm": "Decomposição de 'manter': transação que recebe dados de fora da fronteira para incluir registro no ALI Fornecedores.",
      "nome_processo": "Incluir Fornecedor",
      "classificacao": "EE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "mantenha o cadastro de fornecedores",
      "raciocinio_cpm": "Decomposição de 'manter': transação que recebe dados de fora da fronteira para atualizar registro existente no ALI Fornecedores.",
      "nome_processo": "Alterar Fornecedor",
      "classificacao": "EE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "mantenha o cadastro de fornecedores",
      "raciocinio_cpm": "Decomposição de 'manter': transação que exclui registro existente no ALI Fornecedores.",
      "nome_processo": "Excluir Fornecedor",
      "classificacao": "EE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "mantenha o cadastro de fornecedores",
      "raciocinio_cpm": "Decomposição de 'manter': consulta implícita para recuperar e visualizar os dados cadastrais do fornecedor sem cálculos.",
      "nome_processo": "Consultar Fornecedor",
      "classificacao": "CE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "emita o relatório consolidado exibindo o cálculo do total gasto por fornecedor",
      "raciocinio_cpm": "Apresenta informações para fora da fronteira contendo obrigatoriamente cálculo matemático (total gasto).",
      "nome_processo": "Emitir Relatório de Gastos por Fornecedor",
      "classificacao": "SE",
      "confianca": "Alta"
    }
  ]
}

### EXEMPLO 2 (Anti-Fragmentação: Passos de Tela e Variações fazem parte de 1 único PE + Extração do ALI):
[Trecho do Requisito]: "Registrar Avaliação: 1. O professor consulta a turma desejada; 2. Escolhe a opção notas; 3. Seleciona o aluno na lista; 4. Escolhe entre avaliação simples ou composta (na composta, adiciona sub-itens e o sistema soma a nota total); 5. O sistema valida os campos e registra a nota do aluno."
[Saída Esperada]:
{
  "analise_preliminar_do_trecho": "Entidade de dados mantida: Notas (ALI). Os passos 1 a 5 (selecionar turma, escolher aluno na lista, escolher entre variação simples ou composta com soma e validar campos) são passos e variações de lógica de uma ÚNICA transação autocontida: Registrar Nota. Sub-passos não são contados separadamente.",
  "componentes": [
    {
      "trecho_referencia": "registra a nota do aluno",
      "raciocinio_cpm": "Grupo lógico de dados (Notas) armazenado e mantido dentro da fronteira da aplicação pelo processo de registro.",
      "nome_processo": "Notas",
      "classificacao": "ALI",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "Registrar Avaliação: ... Escolhe entre avaliação simples ou composta ... O sistema valida os campos e registra a nota",
      "raciocinio_cpm": "Transação completa e autocontida iniciada pelo usuário com intenção primária de manter o ALI de Notas. A escolha entre simples/composta e a soma interna são variações de lógica de processamento dentro da mesma EE.",
      "nome_processo": "Registrar Nota",
      "classificacao": "EE",
      "confianca": "Alta"
    }
  ]
}

### EXEMPLO 3 (Importação e Exportação entre Sistemas + Atomização de Listas):
[Trecho do Requisito]: "O sistema deve importar os dados de Turmas a partir do Sistema Central e exportar os registros de Aulas e Faltas para o Sistema Central."
[Saída Esperada]:
{
  "analise_preliminar_do_trecho": "Há três grupos de dados (Turmas, Aulas, Faltas -> ALIs) e três processos de sincronização em lote que devem ser extraídos individualmente (proibido agrupar com vírgulas). Importar dados que entram na fronteira é EE; Exportar dados que saem da fronteira sem cálculo é CE.",
  "componentes": [
    {
      "trecho_referencia": "importar os dados de Turmas a partir do Sistema Central",
      "raciocinio_cpm": "Grupo lógico de dados de Turmas armazenado internamente na aplicação após a carga.",
      "nome_processo": "Turmas",
      "classificacao": "ALI",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "importar os dados de Turmas a partir do Sistema Central",
      "raciocinio_cpm": "Processo elementar que recebe dados vindos de fora da fronteira (Sistema Central) para manter o ALI de Turmas.",
      "nome_processo": "Importar Turmas",
      "classificacao": "EE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "exportar os registros de Aulas e Faltas",
      "raciocinio_cpm": "Envio de dados de Aulas para fora da fronteira (Sistema Central) por recuperação simples sem cálculo matemático.",
      "nome_processo": "Exportar Aulas",
      "classificacao": "CE",
      "confianca": "Alta"
    },
    {
      "trecho_referencia": "exportar os registros de Aulas e Faltas",
      "raciocinio_cpm": "Envio de dados de Faltas para fora da fronteira (Sistema Central) por recuperação simples sem cálculo matemático.",
      "nome_processo": "Exportar Faltas",
      "classificacao": "CE",
      "confianca": "Alta"
    }
  ]
}

### EXEMPLO 4 (Requisitos Não Funcionais, Login, Hardware e Formatação Física - REJEITAR):
[Trecho do Requisito]: "O sistema deve solicitar login e senha de 6 dígitos, sincronizar em até 30 segundos, gerenciar plugins e cookies, e gerar arquivos em formato ASCII alinhados à direita com zeros à esquerda."
[Saída Esperada]:
{
  "analise_preliminar_do_trecho": "O trecho contém exclusivamente Requisitos Não Funcionais: controle de acesso/login técnico, desempenho (30 segundos), infraestrutura (plugins/cookies) e formatação física de implementação (ASCII/zeros à esquerda). Nenhum CFB funcional deve ser extraído.",
  "componentes": []
}
"""

template_decisao = """<|im_start|>system
Você é um Agente Triador CFPS Sênior. Avalie o [Trecho do Requisito] considerando o [Contexto Global].

DIRETRIZES DE TRIAGEM:
1. Primeiro preencha 'raciocinio_triagem' avaliando se há funções de negócio reais ou apenas requisitos técnicos/não funcionais (logs, auditoria automática, layout, mensagens de erro, performance, cookies, plugins, sessão navegador).
2. Se houver apenas Requisitos Não Funcionais, defina 'possui_requisito_funcional' como false.
3. Se houver transações de negócio e você tiver dúvida técnica sobre como aplicar o CPM (ex: fronteira, cálculos implícitos entre SE e CE), defina 'precisa_consultar_manual' como true e formule a pergunta em 'query_busca_manual'. Para CRUDs óbvios, defina 'precisa_consultar_manual' como false.

{format_instructions}<|im_end|>
<|im_start|>user
[Contexto Global]:
{contexto_global}

[Trecho do Requisito]:
{requisito}<|im_end|>
<|im_start|>assistant
"""

template_extracao_json = """<|im_start|>system
Você é um Especialista Certificado em Pontos de Função (CFPS) sênior e rigoroso.
Sua tarefa é analisar o [Trecho do Requisito] e extrair todos os Componentes Funcionais Básicos (Funções de Dados: ALI, AIE e Funções de Transação: EE, SE, CE) válidos.

DIRETRIZES CRÍTICAS OBRIGATÓRIAS:
1. REGRA DA DUALIDADE (TRANSAÇÃO -> DADOS): Nenhuma transação opera no vácuo! Sempre que você identificar uma transação que grava/mantém dados (ex: Registrar Aula, Registrar Nota, Registrar Falta, Importar Turma) ou consulta dados, você DEVE extrair também o Arquivo Lógico correspondente (ex: 'Aulas', 'Notas', 'Faltas', 'Turmas') como 'ALI' (se mantido no sistema) ou 'AIE' (se apenas lido de outro sistema).
2. PROIBIDO AGRUPAR COM VÍRGULAS (ATOMIZAÇÃO): Cada objeto na lista 'componentes' deve representar APENAS UM CFB. Se o texto citar "Registrar Aulas, Faltas e Notas", crie 3 objetos 'EE' separados ('Registrar Aula', 'Registrar Falta', 'Registrar Nota') e 3 objetos 'ALI' separados ('Aulas', 'Faltas', 'Notas').
3. VERBOS GUARDA-CHUVA: Quando encontrar verbos que denotam um CRUD implícito ou salvamento de estado (Manter, Registrar, Cadastrar, Gerenciar), extraia-os como UMA ÚNICA transação (EE) utilizando exatamente o verbo lido (ex: "Registrar Aula"). O motor de regras posterior se encarregará de fragmentá-los em Incluir e Alterar. Apenas extraia Consultar (CE) ou Excluir (EE) se o requisito citar explicitamente essas ações (ex: "consultar turma", "deletar aluno").
4. ANTI-FRAGMENTAÇÃO (TESTE DA TRANSAÇÃO COMPLETA): Um Processo Elementar é a menor unidade significativa que é autocontida e deixa o negócio em estado consistente. NÃO extraia passos intermediários de navegação, seleção prévia (ex: "consultar turma desejada" antes de registrar nota, "selecionar aluno na lista", "clicar no botão confirmar"), validações de campos ou variações de tela (ex: "avaliação simples vs composta") como PEs separados; eles fazem parte do PE principal.
5. REGRAS DE CLASSIFICAÇÃO (IFPUG):
   - ALI: Grupo lógico de dados mantido (incluído, alterado, excluído ou carregado via importação) dentro da fronteira da aplicação.
   - AIE: Grupo lógico de dados apenas consultado/referenciado de outra aplicação externa, sem ser mantido localmente.
   - EE: Recebe dados de fora da fronteira (digitados pelo usuário ou via arquivo de Importação) para manter um ALI ou alterar o comportamento do sistema. (Atenção: Importar dados é SEMPRE EE, nunca SE!).
   - SE: Apresenta dados para fora da fronteira contendo obrigatoriamente cálculos matemáticos na saída, dados derivados ou atualização de ALI.
   - CE: Apresenta ou exporta dados para fora da fronteira por recuperação simples, sem cálculos matemáticos ou atualização de ALI.
6. REJEITE REQUISITOS NÃO FUNCIONAIS: Login/logout, senhas, cookies, plugins, tempos de resposta, especificações de hardware e regras físicas de preenchimento de campos (ASCII, tamanho em bytes, zeros à esquerda) NÃO são contados.

EXEMPLOS DE COMPORTAMENTO ESPERADO (FEW-SHOT):
{exemplos_few_shot}

[Contexto Global do Sistema]:
{contexto_global}

[Regras do Manual CPM Consultadas]:
{regras_cpm}

Instruções de Esquema JSON:
{format_instructions}<|im_end|>
<|im_start|>user
[Trecho do Requisito]:
{requisito}

Responda ESTRITAMENTE com o objeto JSON válido.<|im_end|>
<|im_start|>assistant
"""

def limpar_bloco_json(texto_bruto: str) -> str:
    """Extrai apenas o bloco JSON caso o modelo adicione marcações markdown ao redor."""
    padrao = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
    return padrao.group(0) if padrao else texto_bruto


In [10]:
def executar_agente_apf(trecho_req: str, contexto_global: str) -> Optional[ListaExtracao]:
    # Triagem Autônoma com CoT
    prompt_1 = PromptTemplate(
        template=template_decisao,
        input_variables=["contexto_global", "requisito"],
        partial_variables={"format_instructions": parser_decisao.get_format_instructions()}
    ).format(contexto_global=contexto_global, requisito=trecho_req)

    resposta_decisao_bruta = llm_base.invoke(prompt_1)
    try:
        decisao: DecisaoAgente = parser_decisao.parse(limpar_bloco_json(resposta_decisao_bruta))
    except Exception:
        decisao = DecisaoAgente(
            raciocinio_triagem="Fallback de segurança ativado.",
            possui_requisito_funcional=True,
            precisa_consultar_manual=False
        )

    if not decisao.possui_requisito_funcional:
        return None  # Rejeita Requisitos Não Funcionais na triagem

    # Tool Calling Autônomo (Consulta ao ChromaDB sob demanda)
    if decisao.precisa_consultar_manual and decisao.query_busca_manual:
        print(f"   🔍 [Tool Call] Agente consultou o CPM: '{decisao.query_busca_manual}'")
        regras_recuperadas = consultar_manual_cpm.invoke(decisao.query_busca_manual)
    else:
        regras_recuperadas = "Nenhuma consulta extra necessária. Aplicar regras padrão de intenção primária do CPM."

    # Extração Few-Shot + CoT Validada com Pydantic
    prompt_2 = PromptTemplate(
        template=template_extracao_json,
        input_variables=["contexto_global", "regras_cpm", "requisito"],
        partial_variables={
            "exemplos_few_shot": EXEMPLOS_FEW_SHOT_JSON,
            "format_instructions": parser_extracao.get_format_instructions()
        }
    ).format(
        contexto_global=contexto_global,
        regras_cpm=regras_recuperadas,
        requisito=trecho_req
    )

    resposta_extracao_bruta = llm_base.invoke(prompt_2)
    resultado_validado: ListaExtracao = parser_extracao.parse(limpar_bloco_json(resposta_extracao_bruta))
    return resultado_validado

In [11]:
def expandir_verbos_guarda_chuva(cfb: ComponenteFuncional, id_trecho: int) -> List[dict]:
    """Desdobra verbos guarda-chuva (Registrar, Manter, Cadastrar, Gerenciar) em Incluir e Alterar."""
    pf_estimado = PESOS_NESMA_ESTIMATIVA.get(cfb.classificacao, 0)
    nome_cfb_lower = cfb.nome_processo.lower().strip()

    if cfb.classificacao == "EE" and any(nome_cfb_lower.startswith(v) for v in ["registrar", "manter", "cadastrar", "gerenciar"]):
        entidade = re.sub(r'^(registrar|manter|cadastrar|gerenciar)\s+', '', cfb.nome_processo, flags=re.IGNORECASE)
        return [
            {
                "ID_Trecho": id_trecho,
                "Nome_CFB": f"Incluir {entidade.capitalize()}",
                "Tipo": "EE",
                "PF_NESMA": pf_estimado,
                "Confianca": cfb.confianca,
                "Raciocinio_CPM": cfb.raciocinio_cpm + " (Fragmentado automaticamente pelo motor de regras).",
                "Trecho_Referencia": cfb.trecho_referencia
            },
            {
                "ID_Trecho": id_trecho,
                "Nome_CFB": f"Alterar {entidade.capitalize()}",
                "Tipo": "EE",
                "PF_NESMA": pf_estimado,
                "Confianca": cfb.confianca,
                "Raciocinio_CPM": cfb.raciocinio_cpm + " (Fragmentado automaticamente pelo motor de regras).",
                "Trecho_Referencia": cfb.trecho_referencia
            }
        ]
    else:
        return [{
            "ID_Trecho": id_trecho,
            "Nome_CFB": cfb.nome_processo,
            "Tipo": cfb.classificacao,
            "PF_NESMA": pf_estimado,
            "Confianca": cfb.confianca,
            "Raciocinio_CPM": cfb.raciocinio_cpm,
            "Trecho_Referencia": cfb.trecho_referencia
        }]


In [12]:
def consolidar_dataframe_apf(registros: List[dict]) -> pd.DataFrame:
    """Remove duplicatas semânticas geradas por overlap de chunks."""
    df = pd.DataFrame(registros)
    if df.empty:
        return df

    df["Chave_Unicidade"] = (
        df["Nome_CFB"]
        .str.lower()
        .str.strip()
        .str.replace(r"^(incluir|adicionar|cadastrar)\b", "registrar", regex=True)
        .str.replace(r"s\b", "", regex=True)
    )
    df = (
        df.drop_duplicates(subset=["Chave_Unicidade", "Tipo"], keep="last")
        .drop(columns=["Chave_Unicidade"])
        .reset_index(drop=True)
    )
    return df

In [13]:
# === CÉLULA 8: TESTE DE SANIDADE (WARM-UP DO MODELO) ===
print("\n--- Aquecendo o motor de IA (Teste Rápido de Sanidade da Primeira Inferênia para otimizar o tempo) ---")
trecho_teste = "O sistema deve permitir ao professor registrar as notas dos alunos da turma."

try:
    res_teste = executar_agente_apf(trecho_teste, contexto_global_sistema)
    if res_teste and res_teste.componentes:
        print(f"✅ Motor operacional! {len(res_teste.componentes)} CFBs identificados no teste base.")
    else:
        print("⚪ Teste base retornou vazio (Requisito Não Funcional).")
except Exception as e:
    print(f"⚠️ Erro no aquecimento: {e}")

print("\n🚀 Preparando Interface Gráfica Interativa (Gradio)...")


--- Aquecendo o motor de IA (Teste Rápido de Sanidade da Primeira Inferênia para otimizar o tempo) ---
✅ Motor operacional! 2 CFBs identificados no teste base.

🚀 Preparando Interface Gráfica Interativa (Gradio)...


In [14]:
def processar_entrada_gradio(texto_direto: str, arquivo_upload) -> tuple[str, pd.DataFrame, Optional[str]]:
    """Processa texto colado ou arquivo PDF/TXT enviado pelo usuário na interface Gradio."""
    registros = []

    # upload de um arquivo (.pdf ou .txt)
    if arquivo_upload is not None:
        caminho_arq = arquivo_upload.name if hasattr(arquivo_upload, "name") else str(arquivo_upload)
        texto_completo, splits = carregar_e_fatiar_documento(caminho_arq)
        contexto = gerar_contexto_global(texto_completo)
        trechos = splits[3:] if len(splits) > 4 else splits

        for idx, chunk in enumerate(trechos):
            try:
                res = executar_agente_apf(chunk.page_content, contexto)
                if res and res.componentes:
                    for cfb in res.componentes:
                        registros.extend(expandir_verbos_guarda_chuva(cfb, id_trecho=idx))
            except Exception:
                continue

    # digitou/colou um trecho de requisito diretamente na caixa de texto
    elif texto_direto and len(texto_direto.strip()) >= 10:
        contexto = gerar_contexto_global(texto_direto)
        try:
            res = executar_agente_apf(texto_direto, contexto)
            if res and res.componentes:
                for cfb in res.componentes:
                    registros.extend(expandir_verbos_guarda_chuva(cfb, id_trecho=0))
        except Exception as e:
            return f"⚠️ Erro ao processar o requisito: {e}", pd.DataFrame(), None
    else:
        return "⚠️ Insira um texto de requisito válido ou faça upload de um arquivo (.pdf / .txt).", pd.DataFrame(), None

    df_final = consolidar_dataframe_apf(registros)

    if df_final.empty:
        return "⚪ Nenhum Processo Elementar identificado (Apenas Requisitos Não Funcionais / Regras Técnicas).", pd.DataFrame(), None

    total_pf = int(df_final["PF_NESMA"].sum())
    qtd_cfbs = len(df_final)
    resumo_md = f"### 🎯 Tamanho Funcional Estimado: **{total_pf} PF (NESMA)** | Total de CFBs: **{qtd_cfbs}**"

    # Salva JSON para download direto na interface
    caminho_export = os.path.join(OUTPUT_DIR, "contagem_gradio_export.json")
    df_final.to_json(caminho_export, orient="records", force_ascii=False, indent=2)

    return resumo_md, df_final, caminho_export

In [15]:
# Construção do Layout no Gradio
with gr.Blocks(theme=gr.themes.Soft(), title="Agente APF - IFPUG CPM 4.3.1") as demo:
    gr.Markdown(
        """
        # 📐 Agente Autônomo de Análise de Pontos de Função (IFPUG CPM 4.3.1)
        Extraia e classifique automaticamente **Funções de Transação (EE, SE, CE)** e **Funções de Dados (ALI, AIE)** com estimativa de Pontos de Função (NESMA).
        O sistema combina **Agentic RAG (ChromaDB)** + **Qwen 2.5 (7B Instruct em 8-bits)** + **Motor Neuro-Simbólico de Regras**.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            txt_input = gr.Textbox(
                lines=7,
                label="Opção 1: Cole um Trecho de Especificação de Requisitos",
                placeholder="Ex: O sistema deve permitir ao professor registrar notas (avaliação simples ou composta), registrar faltas e consultar turmas. O tempo de resposta deve ser inferior a 2 segundos."
            )
            file_input = gr.File(
                label="Opção 2: Ou faça Upload de um Documento Completo (.pdf ou .txt)",
                file_types=[".pdf", ".txt"]
            )
            btn_executar = gr.Button("🚀 Executar Contagem de Pontos de Função", variant="primary")

        with gr.Column(scale=2):
            lbl_resumo = gr.Markdown("### Aguardando execução...")
            tabela_output = gr.Dataframe(
                label="Tabela de Componentes Funcionais Básicos (CFBs)",
                interactive=False,
                wrap=True
            )
            arquivo_json_download = gr.File(label="📥 Baixar Resultado Estruturado (.json)")

    gr.Examples(
        examples=[
            ["O sistema deve permitir ao professor consultar turmas, registrar aulas, registrar notas (simples ou composta com cálculo de somatório) e exportar aulas, faltas e notas para o Sistema Acadêmico. O login exige senha de 6 dígitos.", None]
        ],
        inputs=[txt_input, file_input],
        label="Exemplo Rápido (Clique para preencher)"
    )

    btn_executar.click(
        fn=processar_entrada_gradio,
        inputs=[txt_input, file_input],
        outputs=[lbl_resumo, tabela_output, arquivo_json_download]
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7f08863639deb15938.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
